In [1]:
import os
import numpy as np
import csv
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, cross_val_predict

In [2]:
def getImgAndLab(file_path, folder_path):
    """
    This function loads images from a folder, crops them based on annotations in a CSV,
    resizes them, and flattens them for use with Scikit-learn classifiers.
    """
    names = []
    annot = []

    with open(file_path, newline='') as csvfile:
        reader = csv.reader(csvfile)
        for row in reader:
            annot.append(row)

    for entry_name in os.listdir(folder_path):
        if not entry_name.endswith('.csv'):
            names.append(entry_name)

    images = []
    labels = []

    for i in range(len(annot)):
        for j in range(len(names)):
            try:
                # Check if the filename from the CSV matches a file in the directory
                if annot[i][0] == names[j]:
                    path = os.path.join(folder_path, annot[i][0])
                    original_image = Image.open(path)
                    # Bounding box coordinates from the CSV
                    box = (int(annot[i][4]), int(annot[i][5]), int(annot[i][6]), int(annot[i][7]))
                    cropped_image = original_image.crop(box)
                    images.append(cropped_image)
                    labels.append(annot[i][3])
            except (ValueError, IndexError):
                continue

    target_size = (120, 120)
    processed_imgs = []

    for img in images:
        img_resized = img.resize(target_size)
        img_rgb = img_resized.convert('RGB')
        processed_imgs.append(np.array(img_rgb))

    images_np = np.array(processed_imgs, dtype=np.float32) / 255.0

    # --- Key change for Scikit-learn ---
    # Flatten the image data from (n, 120, 120, 3) to (n, 120*120*3)
    num_samples = images_np.shape[0]
    images_flat = images_np.reshape(num_samples, -1)

    # Convert string labels to integers
    unique_labels = sorted(list(set(labels)))
    label_to_int = {label: i for i, label in enumerate(unique_labels)}
    labels_as_integers = [label_to_int[label] for label in labels]
    labels_np = np.array(labels_as_integers)

    return images_flat, labels_np

In [3]:
# Load the datasets using the modified function
print("Loading training data...")
train_images, train_labels = getImgAndLab("./data/train/_annotations.csv", "./data/train/")
print("Loading test data...")
test_images, test_labels = getImgAndLab("./data/test/_annotations.csv", "./data/test/")
print(f"Data loaded. Training data shape: {train_images.shape}, Test data shape: {test_images.shape}")

# Initialize the Random Forest Classifier
# n_estimators is the number of trees in the forest.
# random_state ensures reproducibility.
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1) # n_jobs=-1 uses all available CPU cores

# Train the model
print("\nTraining the Random Forest model...")
model.fit(train_images, train_labels)
print("Training complete.")

Loading training data...
Loading test data...
Data loaded. Training data shape: (4219, 43200), Test data shape: (761, 43200)

Training the Random Forest model...
Training complete.


In [9]:
# Load the validation dataset
print("Loading validation data...")
validation_images, validation_labels = getImgAndLab("./data/valid/_annotations.csv", "./data/valid/")
print(f"Data loaded. Validation data shape: {validation_images.shape}")

# Validate the model
print("\nValidating the Random Forest model...")
scores = cross_val_score(model, validation_images, validation_labels, cv=3, scoring='accuracy')
print(f"Cross-validation scores: {scores}")
print(f"Mean cross-validation accuracy: {scores.mean():.3f}")

Loading validation data...
Data loaded. Validation data shape: (1213, 43200)

Validating the Random Forest model...
Cross-validation scores: [0.89135802 0.89356436 0.89851485]
Mean cross-validation accuracy: 0.894


In [10]:
# Evaluate the model on the test data
print("\nEvaluating the model on the test set...")
predictions = model.predict(test_images)

# Calculate and print the accuracy
accuracy = accuracy_score(test_labels, predictions)
print(f"Test Accuracy: {accuracy:.4f}")

# Print a detailed classification report
print("\nClassification Report:")
print(classification_report(test_labels, predictions))


Evaluating the model on the test set...
Test Accuracy: 0.9566

Classification Report:
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         2
           1       1.00      0.42      0.59        38
           2       0.96      1.00      0.98       610
           3       0.96      0.93      0.94       111

    accuracy                           0.96       761
   macro avg       0.90      0.84      0.83       761
weighted avg       0.96      0.96      0.95       761

